# 02 — Measurement Harness

A uniform scoring system for all three methods:
- **Accuracy**: Relative L2 error against exact solutions
- **Training cost**: Wall-clock time (GPU-synced)
- **Per-instance cost**: Retrain (PINN) vs forward pass (FNO)

The harness also produces the key amortization plot: total cost vs number of instances N.

In [ ]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

print('Harness module loaded.')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

In [ ]:
class MeasurementHarness:
    """
    Reconciled measurement harness for comparing PINN, FNO, and Neural ODE.

    Design decisions:
    - Relative L2 error: scale-invariant, exact against closed-form solutions
    - GPU-synchronized timing: prevents async GPU ops from distorting wall-clock
    - Loss breakdown logged separately: a low total can hide a useless residual term
    - Per-instance std reported for PINNs: training variance is part of the result
    """

    def __init__(self):
        self.registry = {}
        self.device_name = (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
        )

    # ------------------------------------------------------------------
    # Timing
    # ------------------------------------------------------------------
    @staticmethod
    def sync_time():
        """Flush GPU queue before reading wall clock."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return time.perf_counter()

    # ------------------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------------------
    @staticmethod
    def relative_l2_error(u_pred, u_exact):
        """||u_pred - u_exact||_2 / ||u_exact||_2  (epsilon guard for zero fields)."""
        u_pred_flat  = u_pred.flatten().float()
        u_exact_flat = u_exact.flatten().float()
        diff_norm  = torch.linalg.norm(u_pred_flat - u_exact_flat)
        exact_norm = torch.linalg.norm(u_exact_flat)
        return (diff_norm / (exact_norm + 1e-8)).item()

    # ------------------------------------------------------------------
    # Amortized methods (FNO, Neural ODE after training)
    # ------------------------------------------------------------------
    def benchmark_amortized(
        self, name, train_fn, inference_fn, u_exact_test,
        n_test_instances, n_train_samples=1000
    ):
        """
        Profile upfront training + fast per-instance inference.

        Args:
            name: label for registry
            train_fn: callable () -> None  (trains the model in place)
            inference_fn: callable () -> u_pred tensor  (run on all test instances)
            u_exact_test: ground-truth tensor for all test instances
            n_test_instances: number of test instances (for per-instance timing)
            n_train_samples: logged metadata only

        Returns:
            dict of metrics (also stored in self.registry)
        """
        print(f'--- Benchmarking Amortized: {name} (Hardware: {self.device_name}) ---')

        t0 = self.sync_time()
        train_fn()
        train_time = self.sync_time() - t0

        t0 = self.sync_time()
        u_pred = inference_fn()
        total_infer = self.sync_time() - t0
        per_instance = total_infer / max(n_test_instances, 1)

        l2_err = self.relative_l2_error(u_pred, u_exact_test)

        record = {
            'type': 'amortized',
            'train_time_sec': train_time,
            'per_instance_sec': per_instance,
            'per_instance_std': 0.0,
            'n_train_samples': n_train_samples,
            'l2_error': l2_err,
        }
        self.registry[name] = record

        print(f'  Accuracy (Rel L2)  : {l2_err:.6f}')
        print(f'  Train Time         : {train_time:.2f}s')
        print(f'  Per-Instance Time  : {per_instance:.6f}s')
        print()
        return record

    # ------------------------------------------------------------------
    # Instance-based methods (PINN — retrains every time)
    # ------------------------------------------------------------------
    def benchmark_pinn(
        self, name, pinn_factory, u_exact_fn, sample_instances, n_runs=10
    ):
        """
        Profile per-instance training cost and accuracy over multiple runs.

        Args:
            name: label for registry
            pinn_factory: callable (instance_ic) -> (u_pred tensor, loss_dict)
            u_exact_fn: callable (instance_ic) -> u_exact tensor
            sample_instances: list of IC descriptors (length >= n_runs)
            n_runs: number of independent training runs (for statistics)

        Returns:
            dict of metrics
        """
        print(f'--- Benchmarking PINN: {name} over {n_runs} runs ---')

        runtimes = []
        errors   = []
        loss_breakdowns = []

        for i in range(n_runs):
            ic = sample_instances[i]
            u_exact = u_exact_fn(ic)

            t0 = self.sync_time()
            u_pred, loss_dict = pinn_factory(ic)
            rt = self.sync_time() - t0

            err = self.relative_l2_error(u_pred, u_exact)
            runtimes.append(rt)
            errors.append(err)
            loss_breakdowns.append(loss_dict)
            print(f'  Run {i+1}/{n_runs}: time={rt:.2f}s  L2={err:.4f}')

        mean_rt  = float(np.mean(runtimes))
        std_rt   = float(np.std(runtimes))
        mean_err = float(np.mean(errors))

        avg_res = np.mean([d.get('res_loss', 0.0) for d in loss_breakdowns])
        avg_ic  = np.mean([d.get('ic_loss',  0.0) for d in loss_breakdowns])
        avg_bc  = np.mean([d.get('bc_loss',  0.0) for d in loss_breakdowns])

        record = {
            'type': 'instance',
            'train_time_sec': 0.0,
            'per_instance_sec': mean_rt,
            'per_instance_std': std_rt,
            'l2_error': mean_err,
            'loss_breakdown': {'L_res': avg_res, 'L_ic': avg_ic, 'L_bc': avg_bc},
        }
        self.registry[name] = record

        print(f'  Mean Time/Instance : {mean_rt:.2f}s +/- {std_rt:.2f}s')
        print(f'  Mean Rel L2 Error  : {mean_err:.6f}')
        print(f'  Loss Breakdown     -> Residual={avg_res:.3e} | IC={avg_ic:.3e} | BC={avg_bc:.3e}')
        print()
        return record

    # ------------------------------------------------------------------
    # Amortization plot
    # ------------------------------------------------------------------
    def plot_amortization(self, max_instances=1000, log_scale=True, title=None):
        """Total cost = train_time + N * per_instance_time, plotted vs N.

        This is the central figure of the project:
        - PINN: steep straight line (no amortization)
        - FNO: high intercept (train cost) then nearly flat
        - Crossover tells you which method wins at scale.
        """
        N = np.arange(1, max_instances + 1)
        plt.figure(figsize=(10, 6))

        colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
        for (name, metrics), color in zip(self.registry.items(), colors):
            total_cost = metrics['train_time_sec'] + N * metrics['per_instance_sec']
            label = f"{name} (L2={metrics['l2_error']:.4f})"
            plt.plot(N, total_cost, label=label, linewidth=2.5, color=color)

            if metrics['per_instance_std'] > 0:
                std_band = N * metrics['per_instance_std']
                plt.fill_between(
                    N,
                    total_cost - std_band,
                    total_cost + std_band,
                    alpha=0.15, color=color
                )

        # Find crossover points between first two methods
        names = list(self.registry.keys())
        if len(names) >= 2:
            m1 = self.registry[names[0]]
            m2 = self.registry[names[1]]
            denom = m1['per_instance_sec'] - m2['per_instance_sec']
            if abs(denom) > 1e-10:
                crossover = (m2['train_time_sec'] - m1['train_time_sec']) / denom
                if 0 < crossover < max_instances:
                    plt.axvline(x=crossover, color='gray', linestyle=':', linewidth=1.5,
                                label=f'Crossover at N={int(crossover)}')
                    print(f'Break-even: {names[0]} vs {names[1]} at N={crossover:.1f}')

        plt.title(title or f'Amortization Analysis ({self.device_name})', fontsize=14)
        plt.xlabel('Number of Problem Instances (N)', fontsize=12)
        plt.ylabel('Total Wall-Clock Time (seconds)', fontsize=12)
        if log_scale:
            plt.yscale('log')
        plt.grid(True, which='both', linestyle='--', alpha=0.4)
        plt.legend(fontsize=11)
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------
    # Summary table
    # ------------------------------------------------------------------
    def print_summary(self):
        """Print a formatted comparison table."""
        print('=' * 70)
        print(f'{"Method":<25} {"L2 Error":<12} {"Train(s)":<12} {"Per-Inst(s)":<14} {"Type"}')
        print('=' * 70)
        for name, m in self.registry.items():
            std_str = f" +/-{m['per_instance_std']:.2f}" if m['per_instance_std'] > 0 else ''
            print(
                f"{name:<25} {m['l2_error']:<12.6f} {m['train_time_sec']:<12.2f}"
                f" {m['per_instance_sec']:.4f}{std_str:<12} {m['type']}"
            )
        print('=' * 70)


# Instantiate a global harness (will be reused across notebooks if run in sequence)
harness = MeasurementHarness()
print('MeasurementHarness created. Use harness.benchmark_pinn() / harness.benchmark_amortized().')

## Harness Self-Test
Runs a toy comparison to verify timing and error functions work correctly.

In [ ]:
# ---- Self-test with dummy predictors ----
import torch

harness_test = MeasurementHarness()
x_exact = torch.randn(100)

# Perfect predictor
err_perfect = harness_test.relative_l2_error(x_exact, x_exact)
# Noisy predictor
err_noisy   = harness_test.relative_l2_error(x_exact + 0.1 * torch.randn(100), x_exact)
# Constant-zero predictor
err_zero    = harness_test.relative_l2_error(torch.zeros(100), x_exact)

print('=== Harness Self-Test ===')
print(f'Perfect predictor L2 error : {err_perfect:.6f} (expected: 0.0)')
print(f'Noisy predictor L2 error   : {err_noisy:.6f} (expected: ~0.1)')
print(f'Zero predictor L2 error    : {err_zero:.6f} (expected: 1.0)')
print('Self-test passed!' if err_perfect < 1e-6 else 'FAILED: check relative_l2_error')